# REWIND search: type a sentence, get clips back, and score it against the ground truth

Runs **after `1_pipeline.ipynb`** (it needs the clip captions and the SigLIP hand-off in `outputs/batch/`). Writes
`outputs/batch/search/`; look at the results in **`3_explore.ipynb` section 8**. *Run All*; commit **without outputs**.

| Method | Compares the query with | Returns |
|---|---|---|
| `captions` | each clip's caption text (sentence embeddings, `bge-small`) | clips |
| `siglip_selected` | SigLIP embeddings of the frames stage 4 picked (no captions) | frames → their clip |
| `hybrid` | both rankings merged (reciprocal rank fusion); a clip's SigLIP score = its best frame | clips |
| `siglip_1fps` | **baseline**: SigLIP on 1 frame/s of every video (no gating, no captions) | frames |
| `clip_1fps` | **baseline**: the same with CLIP | frames |

**Test queries** (`queries.csv`, written on the first run, edit or add rows freely): every UCA sentence of the
selected UCA videos (answer = its own time span) and one operator-style sentence per MEVA activity type, e.g.
*"a person opens a car door"* (answers = every labelled instance, any camera).

**Scores**: Recall@1/5/10 = share of queries with a correct moment in the top 1/5/10. "Correct" = temporal IoU ≥ 0.3 or
0.5 with an answer (the returned span is a `moment_s` window around the best frame), or **lenient** = the returned clip
overlaps an answer at all. MRR = 1/rank of the first correct one. **reachable %** = queries whose answer overlaps any
clip at all: the rest were dropped by motion gating, and no search can find them. Two settings: **corpus** (search
every video of the dataset) and **video** (only the query's own video, UCA only, the UCA paper's setting).

| Step | Time on the VM (2 h of video) |
|---|---|
| Build the index (caption embeddings) | seconds; `bge-small` downloads once (~130 MB) |
| Baselines (1 frame/s through SigLIP and CLIP, cached) | a few minutes the first time |
| Evaluate all queries | about a minute |

In [ ]:
import os
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

if Path("/mnt/5T_storage").exists():  # team VM: Hugging Face downloads go to the 5T disk
    os.environ.setdefault("HF_HOME", "/mnt/5T_storage/hf_cache")

try:
    import rewind  # noqa: F401  (installed once with `pip install -e .`, see README)
except ImportError:  # not installed: use the repository folder this notebook sits in
    import sys
    sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from rewind import paths
from rewind import search as se

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

## Config

In [ ]:
OUT_ROOT = Path(os.environ.get("REWIND_OUT_ROOT", paths.BATCH))
REMAKE_QUERIES = False     # True = rebuild queries.csv from the ground truth (overwrites your edits)
REBUILD_BASELINES = False  # True = re-embed the 1 frame/s baseline frames even if cached
CFG = {
    "baselines": True,     # False = skip SigLIP / CLIP at 1 frame/s (then only the pipeline's own methods are scored)
    "moment_s": 10.0,      # length of the returned moment around the best frame
    "rrf_k": 60,           # hybrid: rank fusion constant (higher = the two rankings count more equally)
    "top_k": 10,
    "device": "auto",      # "cuda:0" / "cuda:1" if the other group uses one GPU
    "mock": os.environ.get("REWIND_MOCK", "") == "1",   # fake encoders: plumbing check only
}

## 1. Build the index

In [ ]:
t0 = time.time()
index = se.SearchIndex(OUT_ROOT, CFG).build(force=REBUILD_BASELINES)
print(f"ready in {time.time() - t0:.0f} s on {index.device}")
display(index.clips.groupby("dataset").agg(videos=("vdir", "nunique"), clips=("clip_id", "size")))

## 2. Test queries with ground truth
Shown: how many per dataset, and a sample. Your own rows can be added to `outputs/batch/search/queries.csv`
(columns `query_id, dataset, source, text, label, video_id, n_gt, gt`; `gt` = JSON list of `[video_id, start_s, end_s]`).

In [ ]:
queries = se.load_or_make_queries(index, remake=REMAKE_QUERIES)
display(queries.groupby(["dataset", "source"]).agg(queries=("query_id", "size"), answers=("n_gt", "sum")))
display(queries[queries.dataset == "meva"][["query_id", "label", "text", "n_gt"]])
display(queries[queries.dataset == "uca"][["query_id", "video_id", "text"]].head(10))

## 3. Run every query through every method and score it

In [ ]:
t0 = time.time()
results, per_query = se.evaluate(index, queries)
met = se.metrics(per_query)
lat = se.latency_probe(index, queries, n=20)
out = se.save(index, queries, results, per_query, met, lat)
print(f"{len(queries)} queries x {per_query.method.nunique()} methods in {time.time() - t0:.0f} s -> {out}")

## 4. Summary
Main numbers: **R@5 IoU0.3** (strict) and **R@5 lenient** (right clip). Baselines first look at *every* second of
video; the pipeline's methods only see what gating kept, so compare them with **reachable %** in mind.

In [ ]:
main = ["queries", "reachable %", "R@1 IoU0.3", "R@5 IoU0.3", "R@10 IoU0.3", "R@5 IoU0.5",
        "R@1 lenient", "R@5 lenient", "R@5 lenient | reachable", "MRR IoU0.3"]
for (ds, setting), g in met.groupby(["dataset", "setting"], sort=False):
    display(Markdown(f"**{ds}, {setting}** ({'all videos of the dataset' if setting == 'corpus' else 'own video only'})"))
    display(g.set_index("method")[main])
display(Markdown("**Time per query** (encode the text + rank everything; ms)"))
display(lat)

## 5. Try a query
Any sentence. `method`: `hybrid` (default), `captions`, `siglip_selected`, `siglip_1fps`, `clip_1fps`. Filters:
`dataset="meva"|"uca"`, `video_id=…`, `camera_id=…`. The frames are in `3_explore.ipynb` section 8.

In [ ]:
QUERY = "a person gets out of a car"
hits = index.search(QUERY, k=5, method="hybrid")
display(hits[["rank", "dataset", "video_id", "camera_id", "clip_start", "clip_end", "best_t", "score", "summary"]])

## 6. Operator-style queries (no ground truth)
Free-text questions an operator might ask (objective 5): event, person activity, and time conditions. No score: look at
the answers (`3_explore.ipynb` section 8 shows their frames). Saved to `search/demo_results.csv`.

In [ ]:
DEMO_QUERIES = [
    "someone running", "a fight between two people", "a person falls to the ground", "a car crash",
    "a person carrying a bag", "a group of people standing together", "a person on a bicycle",
    "a car parked by the door", "a person in a red shirt", "someone walking at night",
    "a person breaking into a car", "people walking into a building", "a dog", "a person sitting on a bench",
    "smoke or fire", "a truck", "a person using a phone", "two people shaking hands", "a person stealing", "an empty room",
]
demo = pd.concat([index.search(q, k=3, method="hybrid").assign(query=q) for q in DEMO_QUERIES], ignore_index=True)
demo.to_csv(OUT_ROOT / "search" / "demo_results.csv", index=False)
display(demo[demo["rank"] == 1][["query", "dataset", "video_id", "clip_start", "clip_end", "summary"]])